# ___Updating the mycorrhizal states___
--------------------

In [1]:
!python --version

Python 3.13.11


The system cannot find the path specified.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
#--------------------------
# CONSTANTS
#--------------------------

STATE_REPLACEMENTS_TRY = { # THIS IS SPECIFICALLY FOR TRY
    "ECTO": "EcM",
    "Ecto": "EcM",
    "EC": "EcM",
    "ectomycorrhiza": "EcM",
    "Ectomycorrhiza": "EcM",
    "ecto": "EM",
    "ectomycorrhizal": "EcM",
    "ECM": "EcM",
    "vesicular-arbuscular mycorrhiza" : "AM",
    "VAM": "AM",
    "VA": "AM",
    "Non": "NM",
    "AMNM": "NM/AM",
    "Ericoid": "ErM",
    "ERM": "ErM",
    "AM + EM": "AM/EcM",
    "EC/AM": "AM/EcM",
    "Orchid": "OrM",
    "OrM": "OrM"
}

STATE_REPLACEMENTS = { # THIS IS FOR FRED
    "AM + EM": "AM/EcM",
    "AM + EM + NM": "AM/EcM/NM",
    "AM + NM": "AM/NM",
    "AM AM + EM": "AM/EcM",
    "AM AM + EM + NM": "AM/EcM/NM",
    "AM AM + NM": "AM/NM",
    "AM+EM": "AM/EcM",
    "AM+NM": "AM/NM",
    "EM": "EcM",
    "EM AM": "AM/EcM",
    "EM AM + EM + NM": "AM/EcM/NM",
    "EM EM + EEM + NM": "EcM/NM",
    "EM EM+AM": "AM/EcM",
    "ERM": "ErM",
    "EcM-AM": "AM/EcM",
    "NM AM + NM": "AM/NM",
    "NM-AM": "AM/NM",
    "NM/AM": "AM/NM",
}

In [3]:
# https://datadryad.org/dataset/doi:10.5061/dryad.n8bm9
# THE SHEET "Original states data" HAS THE RAW DATA SCRAPED FROM PUBLICATIONS WITHOUT ANY INTEFERENCE FROM THE AUTHORS!!!!
maherali_original = pd.read_excel(r"../../data/chapter2/Maherali.etal.AmNat.Data.xlsx", sheet_name="Original states data", skiprows=range(2),
                                  usecols=("Source", "Original name (Genus species)", "Raw state record from publication"))
maherali_original.rename(mapper={old: old.replace('(', '').replace(')', '').lower().replace(' ', '_') for old in maherali_original.columns}, axis=1, inplace=True) # column names have parentheses and spaces
# taxonomy columns in Maherali et. al. dataset has trailing spaces :(
maherali_original.loc[:, "original_name_genus_species"] = maherali_original.original_name_genus_species.str.strip()
maherali_original.loc[:, "raw_state_record_from_publication"] = maherali_original.raw_state_record_from_publication.str.strip()
maherali_original.drop_duplicates(subset=("original_name_genus_species", "raw_state_record_from_publication"), inplace=True)

# final_maherali = pd.read_excel(r"../../data/chapter2/Maherali.etal.AmNat.Data.xlsx", sheet_name="Final list matched with phylo", skiprows=range(2))
# final_maherali.rename(mapper={old: old.lower().replace(' ', '_') for old in final_maherali.columns}, axis=1, inplace=True)
# final_maherali.genus_species = final_maherali.genus_species.str.strip().str.replace('_', ' ') # the sheet "Final list matched with phylo" has genus and specific epithets concatenated by under scores!

# in TRY, mycorrhiza type is trait id 7
try_myco = pd.read_csv(r"../../data/chapter2/TRY/mycorrhizal_states.txt", delimiter='\t', low_memory=False, encoding="latin1", usecols=["Dataset", "SpeciesName", "AccSpeciesName", "OrigValueStr",
                                "TraitID"]).dropna(subset=["AccSpeciesName", "OrigValueStr", "TraitID"])
# unify the mycorrhizal state info
# 'ECTO', 'NM/AM', 'EC', 'EC/AM', 'AM', 'Ecto', 'Non',        'vesicular-arbuscular mycorrhiza', 'ectomycorrhiza', 'no', '0', 'Ph.th.end.', 'VAM', 'Ectomycorrhiza', 'E.ch.ect.', 'arbuscular',
# 'ec?', 'VA', 'ecto', 'Absent', 'non-ectomycorrhizal', 'ectomycorrhizal', 'Yes', 'No', 'EM', 'AMNM', 'NM', 'AM + EM', 'ERM', 'Ericoid', 'ECM'

try_myco.loc[:, "OrigValueStr"] = try_myco.OrigValueStr.replace(STATE_REPLACEMENTS_TRY)
try_myco = try_myco.query("OrigValueStr.isin(@STATE_REPLACEMENTS_TRY.values())")

# scrape the online only MycoDB metadata and serialize it to the disk
# req = Request(url=r"https://www.nature.com/articles/sdata201628/tables/2", headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:142.0) Gecko/20100101 Firefox/142.0"})
# with urlopen(req) as r:
#     soup = BeautifulSoup(r.read())
# 
# table = soup.find(name="table", attrs={"class": "data last-table"}) # locate the metadata table
# [th.text.strip() for th in table.find_all(name="th")] # column names
# mycodb_descriptions = [[td.text for td in tr.find_all(name="td")] for tr in table.find_all(name="tr")[1:]] # parse the rows
# 
# # create a dataframe using the parsed rows and column names and serialize it to the disk
# pd.DataFrame({ 
#     "Variable": [row[0] for row in mycodb_descriptions],
#     "Description": [row[1] for row in mycodb_descriptions],
#     "Variable Type (range)": [row[2] for row in mycodb_descriptions],
#     "Levels (#studies/level)": [row[3] for row in mycodb_descriptions],
# }).to_csv(r"../data/chapter2/MycoDB_version4_metadata.csv", index=False)

# species that have info in FRED v3 for first order fine root traits RD and SRL
collab_categorical = pd.read_csv(r"../../data/chapter2/FREDv3subset/FRED_subset_collab_categorical.csv")
collab_categorical.insert(column="binominal", loc=0, value=collab_categorical.F01286 + ' ' + collab_categorical.F01287)

# even though we had missing data for photosynthetic pathways and mycorrhizal states in the records that had data for the 4 chosen traits in FRED, FRED could still have info for the categorical traits in other records 
# that were filtered out due to not having all the 4 trait values?????
fred_myco = pd.read_csv(r"../../data/chapter2/FRED/FRED3_Entire_Database_2021.csv", low_memory=False, header=0, skiprows=range(1, 10), encoding="latin1",
                        usecols=("F01286", "F01287", "F00645", "F00004")).dropna(subset=("F01286", "F01287", "F00645")).drop_duplicates()
# concatenate the genus name and specific epithet to introduce a column for binominal name
fred_myco.insert(loc=0, column="binominal", value=fred_myco.F01286.str.strip().str.capitalize() + ' ' + fred_myco.F01287.str.strip().str.lower())

groot_myco = pd.read_csv(r"../../data/chapter2/GRooTFullVersion.csv", low_memory=False, encoding="latin1").query(r"not mycorrhizalAssociationType.isna()").loc[:, ["references", "referencesDataset",
            "genus", "species", "mycorrhizalAssociationType"]].drop_duplicates().dropna(subset=["genus", "species"]).apply(lambda _: _.str.strip(), axis=0)
groot_myco.insert(loc=0, column="binominal", value=groot_myco.genus + ' ' + groot_myco.species)
# even though GRoot has a column for mycorrhizal types directly extracted from FungalRoot (mycorrhizalAssociationTypeFungalRoot), we ignore this because we'll also look through FungalRoot to populate missing values,
# so .....

In [102]:
species_of_interest = collab_categorical.binominal.unique() # loc[:, ["F01286", "F01287"]].drop_duplicates().agg(' '.join, axis=1).str.strip().reset_index(drop=True)
species_of_interest.size # we have 395 species for which we have first order root trait values for SRL and RD :)

395

In [4]:
#------------------------------------------------------------------------------------------
# THE GOAL HERE IS TO FIND THE CORRECT MYCORRHIZAL STATE DATA FOR THESE 395 SPECIES
#------------------------------------------------------------------------------------------

### ___Maherali, H. et al. (2016)___
------------------------------

In [11]:
maherali_original_ = maherali_original.query("original_name_genus_species.isin(@species_of_interest)").drop_duplicates(subset=["original_name_genus_species", "raw_state_record_from_publication"])
maherali_original_

,source,original_name_genus_species,raw_state_record_from_publication
29,Akhmetzhanova et al. 2012,Abies nephrolepis,EM
70,Wang&Qiu2006,Acacia auriculiformis,AM
80,Wang&Qiu2006,Acacia mangium,EM AM
99,Akhmetzhanova et al. 2012,Acer barbinerve,EM
108,Akhmetzhanova et al. 2012,Acer davidii,AM
...,...,...,...
11958,Akhmetzhanova et al. 2012,Ulmus pumila,NM
11985,Wang&Qiu2006,Vaccinium corymbosum,ERM
12240,Akhmetzhanova et al. 2012,Veronica spuria,AM
12444,Wang&Qiu2006,Vitis vinifera,AM


### ___TRY___
--------------------

In [12]:
try_myco_ = try_myco.query("AccSpeciesName.isin(@species_of_interest)").drop_duplicates(subset=["AccSpeciesName", "OrigValueStr"])
try_myco_

,Dataset,SpeciesName,AccSpeciesName,TraitID,OrigValueStr
7,Abisko & Sheffield Database,Caltha palustris,Caltha palustris,7.0,NM/AM
18,Abisko & Sheffield Database,Pinus sylvestris,Pinus sylvestris,7.0,EcM
20,Abisko & Sheffield Database,Populus tremula,Populus tremula,7.0,EcM
33,Abisko & Sheffield Database,Sorbus aucuparia,Sorbus aucuparia,7.0,NM/AM
49,Sheffield Database,Quercus robur,Quercus robur,7.0,EcM
...,...,...,...,...,...
1143409,Independent evolutionary changes in fine-root ...,Castanea henryi,Castanea henryi,7.0,EcM
1143433,Independent evolutionary changes in fine-root ...,Castanopsis wattii,Castanopsis wattii,7.0,EcM
1143481,Independent evolutionary changes in fine-root ...,Lithocarpus chiungchungensis,Lithocarpus chiungchungensis,7.0,EcM
1143583,Independent evolutionary changes in fine-root ...,Quercus serrata,Quercus serrata,7.0,EcM


### ___FRED___
-------------------------------

In [13]:
# again, since this is FRED, look only for species that we do not have mycorrhizal state info for in the subset
missing_states_collab_species = collab_categorical.query("F00645.isna()").loc[:, ["F01286", "F01287"]].drop_duplicates().agg(' '.join, axis=1).str.strip().reset_index(drop=True)
missing_states_collab_species

0             Acer saccharum
1         Fraxinus americana
2            Viola pubescens
3     Hydrophyllum canadense
4             Larix gmelinii
               ...          
91           Ulmus americana
92        Betula platyphylla
93           Malus domestica
94            Prunus persica
95            Vitis vinifera
Length: 96, dtype: object

In [14]:
# some of the records just have "mycorrhizal" for the F00645 column, we do not want that!!!
fred_myco_ = fred_myco.query("binominal.isin(@missing_states_collab_species) and (F00645!='mycorrhizal')").drop_duplicates(subset=["binominal", "F00645"])
fred_myco_

,binominal,F00004,F01286,F01287,F00645
0,Dicranopteris linearis,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",Dicranopteris,linearis,AM
3,Cunninghamia lanceolata,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",Cunninghamia,lanceolata,AM
6,Magnolia baillonii,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",Magnolia,baillonii,AM
11,Acacia auriculiformis,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",Acacia,auriculiformis,AM
15,Gordonia axillaris,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",Gordonia,axillaris,AM
...,...,...,...,...,...
53506,Vitis vinifera,"Akhmetzhanova AA, Soudzilovskaiana NA, Onipche...",Vitis,vinifera,AM
54225,Acer pictum,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",Acer,pictum,AM
54266,Castanopsis faberi,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",Castanopsis,faberi,EM
54281,Elaeocarpus sylvestris,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",Elaeocarpus,sylvestris,AM


### ___GRoot___
----------------------------

In [15]:
groot_myco_ = groot_myco.query("binominal.isin(@species_of_interest)").drop_duplicates(subset=["binominal", "mycorrhizalAssociationType"])
groot_myco_

,binominal,references,referencesDataset,genus,species,mycorrhizalAssociationType
245,Acer negundo,"Adams TS, McCormack ML, Eissenstat DM. 2013. F...",NaN,Acer,negundo,AM
260,Liriodendron tulipifera,"Adams TS, McCormack ML, Eissenstat DM. 2013. F...",NaN,Liriodendron,tulipifera,AM
275,Populus tremuloides,"Adams TS, McCormack ML, Eissenstat DM. 2013. F...",NaN,Populus,tremuloides,AM
3310,Acer davidii,"Chen W, Zeng H, Eissenstat DM, Guo D. 2013. Va...",NaN,Acer,davidii,AM
3334,Acer truncatum,"Chen W, Zeng H, Eissenstat DM, Guo D. 2013. Va...",NaN,Acer,truncatum,EcM-AM
...,...,...,...,...,...,...
87258,Salix viminalis,"Asem A. Akhmetzhanova, Nadejda A. Soudzilovska...",162_Mycorrhizal Intensity Database Across the ...,Salix,viminalis,EcM
87412,Sorbus aucuparia,"Asem A. Akhmetzhanova, Nadejda A. Soudzilovska...",162_Mycorrhizal Intensity Database Across the ...,Sorbus,aucuparia,AM
89241,Crataegus pinnatifida,"Asem A. Akhmetzhanova, Nadejda A. Soudzilovska...",162_Mycorrhizal Intensity Database Across the ...,Crataegus,pinnatifida,AM
89424,Pinus edulis,"Asem A. Akhmetzhanova, Nadejda A. Soudzilovska...",162_Mycorrhizal Intensity Database Across the ...,Pinus,edulis,EcM


In [16]:
#-----------------------------------------------------------
# NOW COMBINE ALL OF THESE TO CREATE A SINGLE DATASET
#-----------------------------------------------------------

In [17]:
data = pd.merge(left=maherali_original_.rename({"original_name_genus_species": "binominal"}, axis=1), left_on="binominal",
         right=try_myco_.rename({"AccSpeciesName": "binominal"}, axis=1), right_on="binominal", how="outer").drop(["SpeciesName", "TraitID"], axis=1)
data

,source,binominal,raw_state_record_from_publication,Dataset,OrigValueStr
0,Akhmetzhanova et al. 2012,Abies nephrolepis,EM,Mycorrhizal Intensity Database Across the Form...,EcM
1,Akhmetzhanova et al. 2012,Abies nephrolepis,EM,FRED - Fine Root Ecology Database,EM
2,Wang&Qiu2006,Acacia auriculiformis,AM,Global 15N Database,AM
3,Wang&Qiu2006,Acacia auriculiformis,AM,Mycorrhiza Database,EcM
4,NaN,Acacia crassicarpa,NaN,FRED - Fine Root Ecology Database,AM/EcM
...,...,...,...,...,...
544,Wang&Qiu2006,Vaccinium corymbosum,ERM,Independent evolutionary changes in fine-root ...,ErM
545,NaN,Vaccinium mandarinorum,NaN,FRED - Fine Root Ecology Database,ErM
546,Akhmetzhanova et al. 2012,Veronica spuria,AM,Mycorrhizal Intensity Database Across the Form...,AM
547,Wang&Qiu2006,Vitis vinifera,AM,FRED - Fine Root Ecology Database,AM


In [18]:
data = pd.merge(left=data, left_on="binominal", right=fred_myco_, right_on="binominal", how="outer").drop(["F01286", "F01287"], axis=1)
data

,source,binominal,raw_state_record_from_publication,Dataset,OrigValueStr,F00004,F00645
0,Akhmetzhanova et al. 2012,Abies nephrolepis,EM,Mycorrhizal Intensity Database Across the Form...,EcM,NaN,NaN
1,Akhmetzhanova et al. 2012,Abies nephrolepis,EM,FRED - Fine Root Ecology Database,EM,NaN,NaN
2,Wang&Qiu2006,Acacia auriculiformis,AM,Global 15N Database,AM,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",AM
3,Wang&Qiu2006,Acacia auriculiformis,AM,Mycorrhiza Database,EcM,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",AM
4,NaN,Acacia crassicarpa,NaN,FRED - Fine Root Ecology Database,AM/EcM,NaN,NaN
...,...,...,...,...,...,...,...
576,Wang&Qiu2006,Vaccinium corymbosum,ERM,Independent evolutionary changes in fine-root ...,ErM,NaN,NaN
577,NaN,Vaccinium mandarinorum,NaN,FRED - Fine Root Ecology Database,ErM,NaN,NaN
578,Akhmetzhanova et al. 2012,Veronica spuria,AM,Mycorrhizal Intensity Database Across the Form...,AM,NaN,NaN
579,Wang&Qiu2006,Vitis vinifera,AM,FRED - Fine Root Ecology Database,AM,"Akhmetzhanova AA, Soudzilovskaiana NA, Onipche...",AM


In [19]:
data = pd.merge(left=data, left_on="binominal", right=groot_myco_, right_on="binominal", how="outer").drop(["genus", "species"], axis=1)
data

,source,binominal,raw_state_record_from_publication,Dataset,OrigValueStr,F00004,F00645,references,referencesDataset,mycorrhizalAssociationType
0,NaN,Abelia biflora,NaN,NaN,NaN,NaN,NaN,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",NaN,AM
1,Akhmetzhanova et al. 2012,Abies nephrolepis,EM,Mycorrhizal Intensity Database Across the Form...,EcM,NaN,NaN,"Dong L, Mao Z, Sun T. 2016. Condensed tannin e...",NaN,EcM
2,Akhmetzhanova et al. 2012,Abies nephrolepis,EM,FRED - Fine Root Ecology Database,EM,NaN,NaN,"Dong L, Mao Z, Sun T. 2016. Condensed tannin e...",NaN,EcM
3,Wang&Qiu2006,Acacia auriculiformis,AM,Global 15N Database,AM,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",AM,"Kong D, Ma C, Zhang Q, Li L, Chen X, Zeng H, G...",NaN,AM
4,Wang&Qiu2006,Acacia auriculiformis,AM,Mycorrhiza Database,EcM,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",AM,"Kong D, Ma C, Zhang Q, Li L, Chen X, Zeng H, G...",NaN,AM
...,...,...,...,...,...,...,...,...,...,...
866,NaN,Veratrum nigrum,NaN,NaN,NaN,NaN,NaN,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",NaN,AM
867,Akhmetzhanova et al. 2012,Veronica spuria,AM,Mycorrhizal Intensity Database Across the Form...,AM,NaN,NaN,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",NaN,AM
868,NaN,Vitis amurensis,NaN,NaN,NaN,NaN,NaN,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",NaN,AM
869,Wang&Qiu2006,Vitis vinifera,AM,FRED - Fine Root Ecology Database,AM,"Akhmetzhanova AA, Soudzilovskaiana NA, Onipche...",AM,NaN,NaN,NaN


In [43]:
# in this 
# source, binominal & raw_state_record_from_publication ARE FROM MAHERALI AT AL
# Dataset, OrigValueStr ARE FROM TRY
# F00004, F00645 ARE FROM FRED V3
# references, referencesDataset & mycorrhizalAssociationType ARE FROM GRoot

In [44]:
# now serialize this :)
# NOTE THAT THIS DOES NOT INCLUDE FUNGALROOT - WHICH IS MESSY AND REQUIRES MANUAL DATA EXTRACTION
data.to_csv(r"../../data/chapter2/FREDv3subset/states_to_fill_FRED_TRY_Maherali_GRoot.csv", index=False)

## ___Conflict resolution & population___
----------------------

In [6]:
lookup = pd.read_csv(r"../../data/chapter2/FREDv3subset/states_to_fill_FRED_TRY_Maherali_GRoot.csv") # the curated compilation dataset

In [7]:
collab_categorical.loc[:, "F00645"] = collab_categorical.F00645.replace(STATE_REPLACEMENTS) # resolve the state encoding heterogeneity before duplicate removal
collab_categorical = collab_categorical.drop_duplicates(subset=["binominal", "F00645"]) # remove duplicates
collab_categorical # 456 RECORDS WITH 395 SPECIES

,binominal,F01286,F01287,F01289,F01290,F00004,F00043,F00645
0,Acer saccharum,Acer,saccharum,Sapindaceae,Sapindales,"Pregitzer KS, Kubiske ME, Yu CK, Hendrick RL. ...",C3,NaN
1,Fraxinus americana,Fraxinus,americana,Oleaceae,Lamiales,"Pregitzer KS, Kubiske ME, Yu CK, Hendrick RL. ...",C3,NaN
2,Viola pubescens,Viola,pubescens,Violaceae,Malpighiales,"Pregitzer KS, Kubiske ME, Yu CK, Hendrick RL. ...",C3,NaN
3,Hydrophyllum canadense,Hydrophyllum,canadense,Boraginaceae,Boraginales,"Pregitzer KS, Kubiske ME, Yu CK, Hendrick RL. ...",C3,NaN
4,Larix gmelinii,Larix,gmelinii,Pinaceae,Pinales,"Wang Z, Guo D, Wang X, Gu J, Mei L. 2006. Fine...",C3,NaN
...,...,...,...,...,...,...,...,...
535,Castanopsis carlesii,Castanopsis,carlesii,Fagaceae,Fagales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",C3,EcM
539,Betula platyphylla,Betula,platyphylla,Betulaceae,Fagales,"Wang Y, Gao G, Wang N, Wang Z, Gu J. 2019. Eff...",C3,NaN
544,Malus domestica,Malus,domestica,Rosaceae,Rosales,"Lavely EL, Chen W, Peterson KA, Klodd AE, Vold...",C3,NaN
545,Prunus persica,Prunus,persica,Rosaceae,Rosales,"Lavely EL, Chen W, Peterson KA, Klodd AE, Vold...",C3,NaN


In [8]:
collab_categorical[collab_categorical.binominal.duplicated(keep=False)].sort_values("binominal") # we do have records with NaNs in column F00645 here!!!

,binominal,F01286,F01287,F01289,F01290,F00004,F00043,F00645
188,Acacia auriculiformis,Acacia,auriculiformis,Fabaceae,Fabales,"Kong D, Ma C, Zhang Q, Li L, Chen X, Zeng H, G...",C3,AM
274,Acacia auriculiformis,Acacia,auriculiformis,Fabaceae,Fabales,"Kong DL, Wang JJ, Kardol P, Wu HF, Zeng H, Den...",C3,NaN
20,Acer caudatum,Acer,caudatum,Sapindaceae,Sapindales,"Shi W, Wang ZQ, Liu JL, Gu JC, Guo DL. 2008. F...",C3,EcM
468,Acer caudatum,Acer,caudatum,Sapindaceae,Sapindales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",C3,AM
404,Acer pictum,Acer,pictum,Sapindaceae,Sapindales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",C3,AM
...,...,...,...,...,...,...,...,...
248,Tilia mandshurica,Tilia,mandshurica,Malvaceae,Malvales,"Chen W, Zeng H, Eissenstat DM, Guo D. 2013. Va...",C3,AM
84,Ulmus americana,Ulmus,americana,Ulmaceae,Rosales,"Valverde-Barrantes OJ, Smemo KA, Blackwood CB...",C3,AM
354,Ulmus americana,Ulmus,americana,Ulmaceae,Rosales,Valverde et al (unpublished),C3,NaN
27,Ulmus davidiana,Ulmus,davidiana,Ulmaceae,Rosales,"Shi W, Wang ZQ, Liu JL, Gu JC, Guo DL. 2008. F...",C3,EcM


In [149]:
collab_categorical[collab_categorical.binominal.duplicated(keep=False)].query(r"F00645.isna()").index # duplicated species names where the duplication happens with NaNs in the state column

Index([  0,   1,   4,   5,  31,  32,  33,  35,  37,  42,  43,  44, 202, 205,
       207, 274, 275, 276, 277, 286, 287, 293, 298, 302, 308, 309, 322, 323,
       324, 325, 326, 327, 328, 329, 331, 332, 333, 334, 336, 337, 339, 340,
       341, 342, 343, 344, 345, 346, 349, 350, 352, 353, 354, 539],
      dtype='int64')

In [16]:
 # get rid of the NaN duplicates
collab_categorical = collab_categorical.drop(index=collab_categorical[collab_categorical.binominal.duplicated(keep=False)].query(r"F00645.isna()").index).sort_values(by="binominal").reset_index(drop=True)
collab_categorical # 402 records with 395 species, not bad

,binominal,F01286,F01287,F01289,F01290,F00004,F00043,F00645
0,Abelia biflora,Abelia,biflora,Caprifoliaceae,Dipsacales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",C3,AM
1,Abies fargesii,Abies,fargesii,Pinaceae,Pinales,"Pu X, Yin C, Xioa Q, Qiao M, Liu Q. 2016. Fine...",C3,NaN
2,Abies nephrolepis,Abies,nephrolepis,Pinaceae,Pinales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",C3,EcM
3,Acacia auriculiformis,Acacia,auriculiformis,Fabaceae,Fabales,"Kong D, Ma C, Zhang Q, Li L, Chen X, Zeng H, G...",C3,AM
4,Acacia crassicarpa,Acacia,crassicarpa,Fabaceae,Fabales,"Kong D, Ma C, Zhang Q, Li L, Chen X, Zeng H, G...",C3,AM/EcM
...,...,...,...,...,...,...,...,...
397,Veronica spuria,Veronica,spuria,Plantaginaceae,Lamiales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",NaN,AM
398,Viola pubescens,Viola,pubescens,Violaceae,Malpighiales,"Pregitzer KS, Kubiske ME, Yu CK, Hendrick RL. ...",C3,NaN
399,Vitis amurensis,Vitis,amurensis,Vitaceae,Vitales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",C3,AM
400,Vitis vinifera,Vitis,vinifera,Vitaceae,Vitales,"Lavely EL, Chen W, Peterson KA, Klodd AE, Vold...",C3,NaN


In [17]:
assert collab_categorical.binominal.unique().size == 395

In [18]:
# this must be executed after the removal of the NaN duplicates!!!!!!!
collab_categorical[collab_categorical.binominal.duplicated(keep=False)] # CONFLICTS WITHIN THE ORIGINAL FRED SUBSET!!!!
# this needs to be resolved

,binominal,F01286,F01287,F01289,F01290,F00004,F00043,F00645
8,Acer caudatum,Acer,caudatum,Sapindaceae,Sapindales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",C3,AM
9,Acer caudatum,Acer,caudatum,Sapindaceae,Sapindales,"Shi W, Wang ZQ, Liu JL, Gu JC, Guo DL. 2008. F...",C3,EcM
14,Acer pictum,Acer,pictum,Sapindaceae,Sapindales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",C3,AM
15,Acer pictum,Acer,pictum,Sapindaceae,Sapindales,"Shi W, Wang ZQ, Liu JL, Gu JC, Guo DL. 2008. F...",C3,EcM
20,Acer tataricum,Acer,tataricum,Sapindaceae,Sapindales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",C3,AM
21,Acer tataricum,Acer,tataricum,Sapindaceae,Sapindales,"Shi W, Wang ZQ, Liu JL, Gu JC, Guo DL. 2008. F...",C3,EcM
29,Adiantum pedatum,Adiantum,pedatum,Pteridaceae,Polypodiales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",C3,mycorrhizal
30,Adiantum pedatum,Adiantum,pedatum,Pteridaceae,Polypodiales,"Dong X, Wang H, Gu J, Wang Y, Wang Z. 2014. Ro...",C3,AM
372,Syringa reticulata,Syringa,reticulata,Oleaceae,Lamiales,"Valverde-Barrantes OJ, Smemo KA, Blackwood CB...",C3,AM
373,Syringa reticulata,Syringa,reticulata,Oleaceae,Lamiales,"Chen W, Zeng H, Eissenstat DM, Guo D. 2013. Va...",C3,AM/EcM


In [19]:
state_conflicts_within_fred = collab_categorical[collab_categorical.binominal.duplicated(keep=False)].binominal.unique() # species that have intra FRED conflicts
state_conflicts_within_fred # just 7 species though

array(['Acer caudatum', 'Acer pictum', 'Acer tataricum',
       'Adiantum pedatum', 'Syringa reticulata', 'Tilia mandshurica',
       'Ulmus davidiana'], dtype=object)

In [20]:
#------------------------------------------------------------------------------------------------------------------------------------
# REMEMBER THAT WE CANNOT JUST DROP THE RECORDS WITHOUT STATE INFO BECAUSE WE HAVE THE ROOT TRAIT INFO FOR THOSE SPECIES
#------------------------------------------------------------------------------------------------------------------------------------

In [23]:
merged = pd.merge(left=collab_categorical, left_on="binominal", right=lookup, right_on="binominal", how="left", suffixes=('_', None))
merged.sort_values(by="binominal", inplace=True)

In [24]:
merged

,binominal,F01286,F01287,F01289,F01290,F00004_,F00043,F00645_,source,raw_state_record_from_publication,Dataset,OrigValueStr,F00004,F00645,references,referencesDataset,mycorrhizalAssociationType
0,Abelia biflora,Abelia,biflora,Caprifoliaceae,Dipsacales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",C3,AM,NaN,NaN,NaN,NaN,NaN,NaN,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",NaN,AM
1,Abies fargesii,Abies,fargesii,Pinaceae,Pinales,"Pu X, Yin C, Xioa Q, Qiao M, Liu Q. 2016. Fine...",C3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Abies nephrolepis,Abies,nephrolepis,Pinaceae,Pinales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",C3,EcM,Akhmetzhanova et al. 2012,EM,Mycorrhizal Intensity Database Across the Form...,EcM,NaN,NaN,"Dong L, Mao Z, Sun T. 2016. Condensed tannin e...",NaN,EcM
3,Abies nephrolepis,Abies,nephrolepis,Pinaceae,Pinales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",C3,EcM,Akhmetzhanova et al. 2012,EM,FRED - Fine Root Ecology Database,EM,NaN,NaN,"Dong L, Mao Z, Sun T. 2016. Condensed tannin e...",NaN,EcM
4,Acacia auriculiformis,Acacia,auriculiformis,Fabaceae,Fabales,"Kong D, Ma C, Zhang Q, Li L, Chen X, Zeng H, G...",C3,AM,Wang&Qiu2006,AM,Global 15N Database,AM,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",AM,"Kong D, Ma C, Zhang Q, Li L, Chen X, Zeng H, G...",NaN,AM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
922,Veronica spuria,Veronica,spuria,Plantaginaceae,Lamiales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",NaN,AM,Akhmetzhanova et al. 2012,AM,Mycorrhizal Intensity Database Across the Form...,AM,NaN,NaN,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",NaN,AM
923,Viola pubescens,Viola,pubescens,Violaceae,Malpighiales,"Pregitzer KS, Kubiske ME, Yu CK, Hendrick RL. ...",C3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
924,Vitis amurensis,Vitis,amurensis,Vitaceae,Vitales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",C3,AM,NaN,NaN,NaN,NaN,NaN,NaN,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",NaN,AM
925,Vitis vinifera,Vitis,vinifera,Vitaceae,Vitales,"Lavely EL, Chen W, Peterson KA, Klodd AE, Vold...",C3,NaN,Wang&Qiu2006,AM,FRED - Fine Root Ecology Database,AM,"Akhmetzhanova AA, Soudzilovskaiana NA, Onipche...",AM,NaN,NaN,NaN


In [25]:
STATE_COLUMNS = ["F00645_", "F00645", "raw_state_record_from_publication", "OrigValueStr", "mycorrhizalAssociationType"] # F00645_ is the column from the FRED subset we have root trait info for

In [28]:
#---------------------------------------------------------------------------------------------------------
# BEFORE CHECKING FOR CONFLICTS, MAKE SURE ALL THE STATE ENCODINGS ARE IN A UNIFIED FORMAT
#---------------------------------------------------------------------------------------------------------

In [30]:
merged.loc[:, STATE_COLUMNS].apply(lambda _: _.dropna().unique()) # HEAPS OF VARIATION IN THE ENCODING

F00645_                                 [AM, EcM, AM/EcM, AM/NM, mycorrhizal, NM, ErM]
F00645                                                           [AM, NM, EM, AM + EM]
raw_state_record_from_publication    [EM, AM, EM AM, NM, AM AM + EM + NM, NM AM + N...
OrigValueStr                                     [EcM, EM, AM, AM/EcM, NM/AM, NM, ErM]
mycorrhizalAssociationType                           [AM, EcM, EcM-AM, NM, NM-AM, ErM]
dtype: object

In [31]:
np.unique(np.concatenate(merged.loc[:, STATE_COLUMNS + ["F00645_"]].apply(lambda _: _.dropna().unique()).values))

array(['AM', 'AM + EM', 'AM + EM + NM', 'AM + NM', 'AM AM + EM',
       'AM AM + EM + NM', 'AM AM + NM', 'AM+EM', 'AM+NM', 'AM/EcM',
       'AM/NM', 'EM', 'EM AM', 'EM AM + EM + NM', 'EM EM + EEM + NM',
       'EM EM+AM', 'ERM', 'EcM', 'EcM-AM', 'ErM', 'NM', 'NM AM + NM',
       'NM-AM', 'NM/AM', 'mycorrhizal'], dtype=object)

In [32]:
merged.loc[:, STATE_COLUMNS] = merged.loc[:, STATE_COLUMNS].apply(lambda col: col.replace(STATE_REPLACEMENTS)) # apply the replacements

In [33]:
# first look at the conflicts between the original FRED subset and the lookup dataset
# collect the indices for species that has more than one unique state in the state columns
conflict_between_idx = np.array([_ for (_, row) in merged.query("not F00645_.isna()").loc[:, STATE_COLUMNS].iterrows() if row.dropna().str.strip().unique().size > 1])

In [34]:
merged.query("not F00645_.isna()").loc[conflict_between_idx, ["binominal"] + STATE_COLUMNS] # FUCK

,binominal,F00645_,F00645,raw_state_record_from_publication,OrigValueStr,mycorrhizalAssociationType
5,Acacia auriculiformis,AM,AM,AM,EcM,AM
7,Acacia crassicarpa,AM/EcM,NaN,NaN,AM/EcM,AM
8,Acacia crassicarpa,AM/EcM,NaN,NaN,AM,AM/EcM
9,Acacia crassicarpa,AM/EcM,NaN,NaN,AM,AM
15,Acacia mangium,AM/EcM,NaN,AM/EcM,AM/EcM,AM
...,...,...,...,...,...,...
912,Ulmus pumila,AM,NaN,EcM,AM,AM
911,Ulmus pumila,AM,NaN,AM,AM/NM,AM
910,Ulmus pumila,AM,NaN,AM,EcM,AM
913,Ulmus pumila,AM,NaN,EcM,EcM,AM


In [36]:
merged.F00645.unique()

array([nan, 'AM', 'NM', 'EcM', 'AM/EcM'], dtype=object)

In [35]:
collab_categorical.F00645.unique()

array(['AM', nan, 'EcM', 'AM/EcM', 'AM/NM', 'mycorrhizal', 'NM', 'ErM'],
      dtype=object)

In [43]:
state_conflicts_between_datasets = merged.query("not F00645_.isna()").loc[conflict_between_idx, :].binominal.unique()
state_conflicts_between_datasets # that's still a LOT!!!

array(['Acacia auriculiformis', 'Acacia crassicarpa', 'Acacia mangium',
       'Acer barbinerve', 'Acer caudatum', 'Acer pictum',
       'Acer platanoides', 'Acer pseudoplatanus', 'Acer saccharum',
       'Acer tataricum', 'Acer tegmentosum', 'Actinidia kolomikta',
       'Adiantum pedatum', 'Alnus hirsuta', 'Alnus mandshurica',
       'Betula costata', 'Caltha palustris', 'Castanea henryi',
       'Castanopsis fissa', 'Castanopsis hystrix', 'Castanopsis wattii',
       'Convallaria majalis', 'Coriaria nepalensis', 'Cornus officinalis',
       'Cotoneaster acutifolius', 'Crataegus pinnatifida',
       'Dicranopteris pedata', 'Eleutherococcus senticosus',
       'Engelhardia roxburghiana', 'Equisetum hyemale',
       'Eucalyptus urophylla', 'Euonymus alatus', 'Euonymus verrucosus',
       'Fraxinus chinensis', 'Fraxinus excelsior', 'Fraxinus mandshurica',
       'Galium aparine', 'Garcinia cowa', 'Gironniera subaequalis',
       'Gleditsia triacanthos', 'Juglans mandshurica', 'Juglans n

In [44]:
state_conflicts_between_datasets.size # 71 species in 424 records????

71

In [46]:
# all conflicts (DOES NOT INCLUDE STATE MISSING SPECIES)
all_state_conflicts = merged.query("binominal.isin(@state_conflicts_between_datasets) or binominal.isin(@state_conflicts_within_fred)").drop_duplicates(subset=STATE_COLUMNS + ["binominal"]).reset_index(drop=True)
all_state_conflicts

,binominal,F01286,F01287,F01289,F01290,F00004_,F00043,F00645_,source,raw_state_record_from_publication,Dataset,OrigValueStr,F00004,F00645,references,referencesDataset,mycorrhizalAssociationType
0,Acacia auriculiformis,Acacia,auriculiformis,Fabaceae,Fabales,"Kong D, Ma C, Zhang Q, Li L, Chen X, Zeng H, G...",C3,AM,Wang&Qiu2006,AM,Global 15N Database,AM,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",AM,"Kong D, Ma C, Zhang Q, Li L, Chen X, Zeng H, G...",NaN,AM
1,Acacia auriculiformis,Acacia,auriculiformis,Fabaceae,Fabales,"Kong D, Ma C, Zhang Q, Li L, Chen X, Zeng H, G...",C3,AM,Wang&Qiu2006,AM,Mycorrhiza Database,EcM,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",AM,"Kong D, Ma C, Zhang Q, Li L, Chen X, Zeng H, G...",NaN,AM
2,Acacia crassicarpa,Acacia,crassicarpa,Fabaceae,Fabales,"Kong D, Ma C, Zhang Q, Li L, Chen X, Zeng H, G...",C3,AM/EcM,NaN,NaN,FRED - Fine Root Ecology Database,AM/EcM,NaN,NaN,"Kong D, Ma C, Zhang Q, Li L, Chen X, Zeng H, G...",NaN,AM/EcM
3,Acacia crassicarpa,Acacia,crassicarpa,Fabaceae,Fabales,"Kong D, Ma C, Zhang Q, Li L, Chen X, Zeng H, G...",C3,AM/EcM,NaN,NaN,FRED - Fine Root Ecology Database,AM/EcM,NaN,NaN,"Valverde-Barrantes O J, Horning A L, Smemo K A...",NaN,AM
4,Acacia crassicarpa,Acacia,crassicarpa,Fabaceae,Fabales,"Kong D, Ma C, Zhang Q, Li L, Chen X, Zeng H, G...",C3,AM/EcM,NaN,NaN,Independent evolutionary changes in fine-root ...,AM,NaN,NaN,"Kong D, Ma C, Zhang Q, Li L, Chen X, Zeng H, G...",NaN,AM/EcM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
424,Ulmus pumila,Ulmus,pumila,Ulmaceae,Rosales,"Shi W, Wang ZQ, Liu JL, Gu JC, Guo DL. 2008. F...",C3,AM,Akhmetzhanova et al. 2012,AM,Mycorrhizal Intensity Database Across the Form...,EcM,NaN,NaN,"Dong L, Mao Z, Sun T. 2016. Condensed tannin e...",NaN,AM
425,Ulmus pumila,Ulmus,pumila,Ulmaceae,Rosales,"Shi W, Wang ZQ, Liu JL, Gu JC, Guo DL. 2008. F...",C3,AM,Akhmetzhanova et al. 2012,AM,Mycorrhizal Intensity Database Across the Form...,AM,NaN,NaN,"Dong L, Mao Z, Sun T. 2016. Condensed tannin e...",NaN,AM
426,Ulmus pumila,Ulmus,pumila,Ulmaceae,Rosales,"Shi W, Wang ZQ, Liu JL, Gu JC, Guo DL. 2008. F...",C3,AM,Akhmetzhanova et al. 2012,EcM,Mycorrhizal Intensity Database Across the Form...,EcM,NaN,NaN,"Dong L, Mao Z, Sun T. 2016. Condensed tannin e...",NaN,AM
427,Vaccinium mandarinorum,Vaccinium,mandarinorum,Ericaceae,Ericales,"Kong D, Ma C, Zhang Q, Li L, Chen X, Zeng H, G...",C3,ErM,NaN,NaN,FRED - Fine Root Ecology Database,ErM,NaN,NaN,"Kong D, Ma C, Zhang Q, Li L, Chen X, Zeng H, G...",NaN,ErM


In [78]:
all_state_conflicts.loc[:, ["binominal"] + STATE_COLUMNS]

,binominal,F00645_,F00645,raw_state_record_from_publication,OrigValueStr,mycorrhizalAssociationType
0,Acacia auriculiformis,AM,AM,AM,AM,AM
1,Acacia auriculiformis,AM,AM,AM,EcM,AM
2,Acacia crassicarpa,AM/EcM,NaN,NaN,AM/EcM,AM/EcM
3,Acacia crassicarpa,AM/EcM,NaN,NaN,AM/EcM,AM
4,Acacia crassicarpa,AM/EcM,NaN,NaN,AM,AM/EcM
...,...,...,...,...,...,...
424,Ulmus pumila,AM,NaN,AM,EcM,AM
425,Ulmus pumila,AM,NaN,AM,AM,AM
426,Ulmus pumila,AM,NaN,EcM,EcM,AM
427,Vaccinium mandarinorum,ErM,NaN,NaN,ErM,ErM


In [80]:
all_state_conflicts.binominal.unique().size # okay

71

In [47]:
# resolve the above conflicts using FungalRoot, manually
all_state_conflicts.to_csv(r"../../data/chapter2/FREDv3subset/all_state_conflicts.csv", index=False)

In [62]:
merged.loc[merged.loc[:, STATE_COLUMNS].isna().sum(axis=1) == len(STATE_COLUMNS)]

,binominal,F01286,F01287,F01289,F01290,F00004_,F00043,F00645_,source,raw_state_record_from_publication,Dataset,OrigValueStr,F00004,F00645,references,referencesDataset,mycorrhizalAssociationType
1,Abies fargesii,Abies,fargesii,Pinaceae,Pinales,"Pu X, Yin C, Xioa Q, Qiao M, Liu Q. 2016. Fine...",C3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
162,Alnus formosana,Alnus,formosana,Betulaceae,Fagales,"Miao Y, Chen YL, Li XW, Fan C, Liu YK, Yang ZJ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
174,Altingia gracilipes,Altingia,gracilipes,Altingiaceae,Saxifragales,"Xiong D, Huang J, Yang Z, Lu Z, Chen G, Yang Y...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
175,Altingia obovata,Altingia,obovata,Altingiaceae,Saxifragales,"Xu Y, Gu JC, Dong XY, Liu Y, Wang ZQ. 2011. Fi...",C3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
186,Ardisia quinquegona,Ardisia,quinquegona,Primulaceae,Ericales,"Wang J-J, Tharayil N, Chow AT, Suseela V, Zeng...",C3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
187,Artemisia halodendron,Artemisia,halodendron,Asteraceae,Asterales,"Huang G, Zhao X-Y, Zhao H-L, Huang Y-X, Zuo X...",C3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
244,Caragana microphylla,Caragana,microphylla,Fabaceae,Fabales,"Huang G, Zhao X-Y, Zhao H-L, Huang Y-X, Zuo X...",C3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
289,Cinnamomum chekiangense,Cinnamomum,chekiangense,Lauraceae,Laurales,"Xiong D, Huang J, Yang Z, Lu Z, Chen G, Yang Y...",C3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
321,Cystopteris sudetica,Cystopteris,sudetica,Cystopteridaceae,Polypodiales,"Dong X, Wang H, Gu J, Wang Y, Wang Z. 2014. Ro...",C3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
419,Hydrophyllum canadense,Hydrophyllum,canadense,Boraginaceae,Boraginales,"Pregitzer KS, Kubiske ME, Yu CK, Hendrick RL. ...",C3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [64]:
merged.binominal.isin(collab_categorical.binominal.unique()).mean()

np.float64(1.0)

In [50]:
# how many species still have no mycorrhizal state info
state_missing_species = merged.binominal.loc[merged.loc[:, STATE_COLUMNS].isna().sum(axis=1) == len(STATE_COLUMNS)].unique()
state_missing_species.size

13

In [49]:
np.savetxt(fname=r"../../data/chapter2/FREDv3subset/missing_states.csv", X=merged.binominal.loc[merged.loc[:, STATE_COLUMNS + ["F00645_"]].isna().sum(axis=1) == len(STATE_COLUMNS) + 1].unique(),
          delimiter=',', fmt=r"%s")

In [ ]:
#--------------------------------------------------------------
# there are 71 species with state conflicts between datasets
# 7 species with state conflicts within the FRED subset
# 13 species with no mycorrhizal state info whatsoever
#--------------------------------------------------------------

In [ ]:
# 91 problematic species

# ___Post-population analyses___
---------------------------

In [21]:
assert collab_categorical.binominal.unique().size == 395
all_state_conflicts = pd.read_csv(r"../../data/chapter2/FREDv3subset/all_state_conflicts.csv")
state_missing_species = pd.read_csv(r"../../data/chapter2/FREDv3subset/missing_states.csv", header=None).to_numpy().ravel()

In [22]:
# species that have conflicting state information (within FRED and between datasets) and species that do not have state info anywhere
problematic_species = np.concat([all_state_conflicts.binominal.unique(), state_missing_species])
problematic_species.size # 84 :)

84

In [55]:
collab_categorical.query(r"not binominal.isin(@problematic_species)")

,binominal,F01286,F01287,F01289,F01290,F00004,F00043,F00645
0,Abelia biflora,Abelia,biflora,Caprifoliaceae,Dipsacales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",C3,AM
2,Abies nephrolepis,Abies,nephrolepis,Pinaceae,Pinales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",C3,EcM
7,Acer campbellii,Acer,campbellii,Sapindaceae,Sapindales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",C3,AM
10,Acer coriaceifolium,Acer,coriaceifolium,Sapindaceae,Sapindales,"Liu B, Li H, Zhu B, Koide RT, Eissenstat DM, G...",NaN,AM
11,Acer davidii,Acer,davidii,Sapindaceae,Sapindales,"Chen W, Zeng H, Eissenstat DM, Guo D. 2013. Va...",C3,AM
...,...,...,...,...,...,...,...,...
396,Veratrum nigrum,Veratrum,nigrum,Melanthiaceae,Liliales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",C3,AM
397,Veronica spuria,Veronica,spuria,Plantaginaceae,Lamiales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",NaN,AM
399,Vitis amurensis,Vitis,amurensis,Vitaceae,Vitales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",C3,AM
400,Vitis vinifera,Vitis,vinifera,Vitaceae,Vitales,"Lavely EL, Chen W, Peterson KA, Klodd AE, Vold...",C3,NaN


In [71]:
# the easiest and cleanest way to combine the subset of FRED that doesn't have any conflicts or missing data and the hand extracted data is to first drop all the records with conflicting and missing states and 
# then append the hand extracted data

collab_categorical_unproblematic = collab_categorical.query(r"not binominal.isin(@problematic_species)").reset_index(drop=True)
collab_categorical_unproblematic

,binominal,F01286,F01287,F01289,F01290,F00004,F00043,F00645
0,Abelia biflora,Abelia,biflora,Caprifoliaceae,Dipsacales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",C3,AM
1,Abies nephrolepis,Abies,nephrolepis,Pinaceae,Pinales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",C3,EcM
2,Acer campbellii,Acer,campbellii,Sapindaceae,Sapindales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",C3,AM
3,Acer coriaceifolium,Acer,coriaceifolium,Sapindaceae,Sapindales,"Liu B, Li H, Zhu B, Koide RT, Eissenstat DM, G...",NaN,AM
4,Acer davidii,Acer,davidii,Sapindaceae,Sapindales,"Chen W, Zeng H, Eissenstat DM, Guo D. 2013. Va...",C3,AM
...,...,...,...,...,...,...,...,...
306,Veratrum nigrum,Veratrum,nigrum,Melanthiaceae,Liliales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",C3,AM
307,Veronica spuria,Veronica,spuria,Plantaginaceae,Lamiales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",NaN,AM
308,Vitis amurensis,Vitis,amurensis,Vitaceae,Vitales,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",C3,AM
309,Vitis vinifera,Vitis,vinifera,Vitaceae,Vitales,"Lavely EL, Chen W, Peterson KA, Klodd AE, Vold...",C3,NaN


In [72]:
collab_categorical_unproblematic.F00645.value_counts(dropna=False) # how come we have 29 NaNs here?????

F00645
AM             230
EcM             37
NaN             29
mycorrhizal      7
NM               3
AM/NM            3
AM/EcM           1
ErM              1
Name: count, dtype: int64

In [74]:
collab_categorical_nan_states = collab_categorical_unproblematic.query(r"F00645.isna()").binominal.unique()

In [76]:
merged.query("binominal.isin(@collab_categorical_nan_states)")

,binominal,F01286,F01287,F01289,F01290,F00004_,F00043,F00645_,source,raw_state_record_from_publication,Dataset,OrigValueStr,F00004,F00645,references,referencesDataset,mycorrhizalAssociationType
43,Acer negundo,Acer,negundo,Sapindaceae,Sapindales,"McCormack ML, Adams TS, Smithwick EAH, Eissens...",C3,NaN,Akhmetzhanova et al. 2012,NM,Mycorrhizal Intensity Database Across the Form...,AM,"Comas LH, Bouma TJ, Eissenstat DM. 2002. Linki...",NM,"Adams TS, McCormack ML, Eissenstat DM. 2013. F...",NaN,AM
44,Acer negundo,Acer,negundo,Sapindaceae,Sapindales,"McCormack ML, Adams TS, Smithwick EAH, Eissens...",C3,NaN,Akhmetzhanova et al. 2012,NM,Mycorrhizal Intensity Database Across the Form...,AM,"Comas LH, Bouma TJ, Eissenstat DM. 2002. Linki...",NM,"Comas LH, Bouma TJ, Eissenstat DM. 2002. Linki...",NaN,NM
45,Acer negundo,Acer,negundo,Sapindaceae,Sapindales,"McCormack ML, Adams TS, Smithwick EAH, Eissens...",C3,NaN,Akhmetzhanova et al. 2012,NM,FRED - Fine Root Ecology Database,AM/NM,"Adams TS, McCormack ML, Eissenstat DM. 2013. F...",AM,"Adams TS, McCormack ML, Eissenstat DM. 2013. F...",NaN,AM
46,Acer negundo,Acer,negundo,Sapindaceae,Sapindales,"McCormack ML, Adams TS, Smithwick EAH, Eissens...",C3,NaN,Akhmetzhanova et al. 2012,NM,FRED - Fine Root Ecology Database,AM/NM,"Adams TS, McCormack ML, Eissenstat DM. 2013. F...",AM,"Comas LH, Bouma TJ, Eissenstat DM. 2002. Linki...",NaN,NM
49,Acer negundo,Acer,negundo,Sapindaceae,Sapindales,"McCormack ML, Adams TS, Smithwick EAH, Eissens...",C3,NaN,Akhmetzhanova et al. 2012,NM,FRED - Fine Root Ecology Database,NM,"Adams TS, McCormack ML, Eissenstat DM. 2013. F...",AM,"Adams TS, McCormack ML, Eissenstat DM. 2013. F...",NaN,AM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
697,Quercus rubra,Quercus,rubra,Fagaceae,Fagales,"McCormack ML, Adams TS, Smithwick EAH, Eissens...",C3,NaN,Wang&Qiu2006,EcM,FRED - Fine Root Ecology Database,NM,"Comas LH, Eissenstat DM. 2009. Patterns in roo...",EcM,"Comas LH, Bouma TJ, Eissenstat DM. 2002. Linki...",NaN,NM
693,Quercus rubra,Quercus,rubra,Fagaceae,Fagales,"McCormack ML, Adams TS, Smithwick EAH, Eissens...",C3,NaN,Wang&Qiu2006,EcM,FRED - Fine Root Ecology Database,EcM,"Comas LH, Eissenstat DM. 2009. Patterns in roo...",EcM,"Comas LH, Bouma TJ, Eissenstat DM. 2002. Linki...",NaN,NM
796,Salix gordejevii,Salix,gordejevii,Salicaceae,Malpighiales,"Huang G, Zhao X-Y, Zhao H-L, Huang Y-X, Zuo X...",C3,NaN,NaN,NaN,Mycorrhiza Database,EcM,NaN,NaN,NaN,NaN,NaN
918,Vaccinium corymbosum,Vaccinium,corymbosum,Ericaceae,Ericales,"Valenzuela-Estrada LR, Vera-Caraballo V, Ruth ...",C3,NaN,Wang&Qiu2006,ErM,Independent evolutionary changes in fine-root ...,ErM,NaN,NaN,NaN,NaN,NaN


In [132]:
conflict = pd.read_excel(r"../../data/chapter2/FREDv3subset/state_conflicts.xlsx", sheet_name="states_resolved") # species that had conflicting state in our compiled lookup dataset
missing = pd.read_excel(r"../../data/chapter2/FREDv3subset/state_conflicts.xlsx", sheet_name="missing_states") # species that did not have state info in our compiled lookup dataset

In [133]:
hand_extracted_data = pd.concat([missing, conflict]) # these states need not to be cross checked with anything, if found conflicting with others, just drop them
hand_extracted_data

,binominal,actual_publication_reports_in_fungalroot,fungalroot_agreed_species_state
0,Abies fargesii,EcM,EcM
1,Alnus formosana,EcM,EcM
2,Altingia gracilipes,NaN,AM
3,Altingia obovata,NaN,AM
4,Ardisia quinquegona,AM/NM,AM
...,...,...,...
62,Tilia mandshurica,EcM,EcM
63,Ulmus davidiana,AM,AM
64,Ulmus laciniata,AM/EcM,AM
65,Ulmus pumila,AM/EcM,AM


80

In [80]:
hand_extracted_data # harvest the taxonomic columns for these species

,binominal,actual_publication_reports_in_fungalroot,fungalroot_agreed_species_state
0,Abies fargesii,EcM,EcM
1,Alnus formosana,EcM,EcM
2,Altingia gracilipes,NaN,AM
3,Altingia obovata,NaN,AM
4,Ardisia quinquegona,AM/NM,AM
...,...,...,...
62,Tilia mandshurica,EcM,EcM
63,Ulmus davidiana,AM,AM
64,Ulmus laciniata,AM/EcM,AM
65,Ulmus pumila,AM/EcM,AM


In [89]:
hand_extracted_data_restructed = pd.merge(left=hand_extracted_data, left_on="binominal", right=collab_categorical, right_on="binominal", how="left").drop_duplicates(subset=["binominal",
                            "actual_publication_reports_in_fungalroot", "fungalroot_agreed_species_state"])
hand_extracted_data_restructed

,binominal,actual_publication_reports_in_fungalroot,fungalroot_agreed_species_state,F01286,F01287,F01289,F01290,F00004,F00043,F00645
0,Abies fargesii,EcM,EcM,Abies,fargesii,Pinaceae,Pinales,"Pu X, Yin C, Xioa Q, Qiao M, Liu Q. 2016. Fine...",C3,NaN
1,Alnus formosana,EcM,EcM,Alnus,formosana,Betulaceae,Fagales,"Miao Y, Chen YL, Li XW, Fan C, Liu YK, Yang ZJ...",NaN,NaN
2,Altingia gracilipes,NaN,AM,Altingia,gracilipes,Altingiaceae,Saxifragales,"Xiong D, Huang J, Yang Z, Lu Z, Chen G, Yang Y...",NaN,NaN
3,Altingia obovata,NaN,AM,Altingia,obovata,Altingiaceae,Saxifragales,"Xu Y, Gu JC, Dong XY, Liu Y, Wang ZQ. 2011. Fi...",C3,NaN
4,Ardisia quinquegona,AM/NM,AM,Ardisia,quinquegona,Primulaceae,Ericales,"Wang J-J, Tharayil N, Chow AT, Suseela V, Zeng...",C3,NaN
...,...,...,...,...,...,...,...,...,...,...
92,Tilia mandshurica,EcM,EcM,Tilia,mandshurica,Malvaceae,Malvales,"Shi W, Wang ZQ, Liu JL, Gu JC, Guo DL. 2008. F...",C3,EM
94,Ulmus davidiana,AM,AM,Ulmus,davidiana,Ulmaceae,Rosales,"Shi W, Wang ZQ, Liu JL, Gu JC, Guo DL. 2008. F...",C3,EM
96,Ulmus laciniata,AM/EcM,AM,Ulmus,laciniata,Ulmaceae,Rosales,"Shi W, Wang ZQ, Liu JL, Gu JC, Guo DL. 2008. F...",C3,EM
97,Ulmus pumila,AM/EcM,AM,Ulmus,pumila,Ulmaceae,Rosales,"Shi W, Wang ZQ, Liu JL, Gu JC, Guo DL. 2008. F...",C3,AM


In [90]:
hand_extracted_data_restructed.loc[:, "F00645"] = hand_extracted_data_restructed.actual_publication_reports_in_fungalroot # update the pathway column with hand extracted data
hand_extracted_data_restructed.loc[:, "F00004"] = "FungalRoot" # update the reference column

In [95]:
hand_extracted_data_restructed = hand_extracted_data_restructed.drop(["actual_publication_reports_in_fungalroot", "fungalroot_agreed_species_state"], axis=1) # drop the unwanted columns

In [99]:
pd.concat([collab_categorical_unproblematic, hand_extracted_data_restructed]).drop_duplicates(subset=["binominal", "F00645"])

,binominal,F01286,F01287,F01289,F01290,F00004,F00043,F00645
14,Prunus sibirica,Prunus,sibirica,Rosaceae,Rosales,"Shi W, Wang ZQ, Liu JL, Gu JC, Guo DL. 2008. F...",C3,AM
21,Alnus hirsuta,Alnus,hirsuta,Betulaceae,Fagales,"Shi W, Wang ZQ, Liu JL, Gu JC, Guo DL. 2008. F...",NaN,EM
23,Betula platyphylla,Betula,platyphylla,Betulaceae,Fagales,"Shi W, Wang ZQ, Liu JL, Gu JC, Guo DL. 2008. F...",C3,EM
25,Quercus mongolica,Quercus,mongolica,Fagaceae,Fagales,"Shi W, Wang ZQ, Liu JL, Gu JC, Guo DL. 2008. F...",C3,EM
49,Carpinus betulus,Carpinus,betulus,Betulaceae,Fagales,"Kubisch P, Hertel D, Leuschner C. 2015. Do ect...",C3,EM
...,...,...,...,...,...,...,...,...
92,Tilia mandshurica,Tilia,mandshurica,Malvaceae,Malvales,FungalRoot,C3,EcM
94,Ulmus davidiana,Ulmus,davidiana,Ulmaceae,Rosales,FungalRoot,C3,AM
96,Ulmus laciniata,Ulmus,laciniata,Ulmaceae,Rosales,FungalRoot,C3,AM/EcM
97,Ulmus pumila,Ulmus,pumila,Ulmaceae,Rosales,FungalRoot,C3,AM/EcM
